# Flexible Neural Network Class

In [30]:
# import packages
import numpy as np
import pandas as pd
print("import success")

import success


In [61]:
class NeuralNetwork:
    def __init__(self, layer_list):
        self.layers = []
        for i in range(len(layer_list) - 1):
            self.layers.append(np.random.randn(layer_list[i + 1], layer_list[i] + 1) * 0.01)
    
    def see_layers(self):
        for layer in self.layers:
            print(layer)

    def forward_pass(self, data_point, target, activation, error, error_derivative):
        output = {}
        output["data"] = data_point
        output["layer 1 sum"] = self.layers[0] @ np.vstack([data_point, [1]])
        output["layer 1 output"] = activation(output["layer 1 sum"])
        for i in range(1, len(self.layers)):
            output[f"layer {i+1} sum"] = self.layers[i] @ np.vstack([output[f"layer {i} output"], [1]])
            output[f"layer {i+1} output"] = activation(output[f"layer {i+1} sum"])
        output["error"] = error(output[f"layer {len(self.layers)} output"], target).item()
        output["error gradient"] = error_derivative(output[f"layer {len(self.layers)} output"], target).item()
        return output
        
    def back_prop(self, fpd, learning_rate, activation_derivative):
        delE_delOlast = fpd["error gradient"]
        delE_delSlast = delE_delOlast * activation_derivative(fpd[f"layer {len(self.layers)} output"])
        delE_delWlast = delE_delSlast * np.vstack([fpd[f"layer {len(self.layers) - 1} output"], [1]]).T
        self.layers[-1] = self.layers[-1] - (learning_rate * delE_delWlast)

        for i in range(len(self.layers) - 1, 1, -1):
            delE_delOi = self.layers[i][:, :-1].T @ delE_delSlast
            delE_delSlast = delE_delOi * activation_derivative(fpd[f"layer {i} output"])
            delE_delWi = delE_delSlast @ np.vstack([fpd[f"layer {i-1} output"], [1]]).T
            self.layers[i-1] = self.layers[i-1] - (learning_rate * delE_delWi)

        delE_delO1 = self.layers[1][:, :-1].T @ delE_delSlast
        delE_delS1 = delE_delO1 * activation_derivative(fpd[f"layer 1 output"])
        delE_delW1 = delE_delS1 @ np.vstack([fpd["data"], [1]]).T
        self.layers[0] = self.layers[0] - (learning_rate * delE_delW1)

    def train(self, epochs, dataset, targets, activation, error, error_derivative, learning_rate, activation_derivative):
        for epoch in range(epochs):
            counter = 1
            for i in range(len(dataset)):
                d = self.forward_pass(dataset[i].reshape(-1, 1), targets[i], activation, error, error_derivative)
                self.back_prop(d, learning_rate, activation_derivative)
                counter += 1
                if counter % 50 == 0 and epoch % 50 == 0:
                    print(f"Error: {d["error"]:.6f}")

In [18]:
df = pd.read_csv("customers.csv")
new_df = df[["Time on App", "Length of Membership", "Yearly Amount Spent"]]
new_df.head()

,Time on App,Length of Membership,Yearly Amount Spent
0,12.655651,4.082621,587.951054
1,11.109461,2.664034,392.204933
2,11.330278,4.104543,487.547505
3,13.717514,3.120179,581.852344
4,12.795189,4.446308,599.406092


## Initialise Activation Function and Derivative

In [57]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(y):
    return y * (1 - y)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

## Initialise Error Function and Derivative

In [20]:
def squared_error(output, target):
    return (output - target) ** 2

def error_derivative(output, target):
    return 2 * (output - target)

## Prepare Data

In [21]:
data_df = new_df.drop(['Yearly Amount Spent'], axis=1)
data_df.head()

,Time on App,Length of Membership
0,12.655651,4.082621
1,11.109461,2.664034
2,11.330278,4.104543
3,13.717514,3.120179
4,12.795189,4.446308


In [78]:
data = data_df.to_numpy()
data_min = np.min(data, axis=0)
data_max = np.max(data, axis=0)
std_data = (data - data_min) / (data_max - data_min)

# targets = sigmoid(df[["Yearly Amount Spent"]].to_numpy())
targets_min = np.min(targets, axis=0)
targets_max = np.max(targets, axis=0)
std_targets = (targets - targets_min) / (targets_max - targets_min)
print(std_data.shape)
print(std_targets.shape)

(500, 2)
(500, 1)


## Model Training

In [82]:
model = NeuralNetwork([2, 10, 10, 5, 1])
model.train(250, std_data, targets, sigmoid, squared_error, error_derivative, 0.01, sigmoid_derivative)

Error: 0.186291
Error: 0.143463
Error: 0.112921
Error: 0.090953
Error: 0.074859
Error: 0.062812
Error: 0.053595
Error: 0.046398
Error: 0.040672
Error: 0.036039
Error: 0.000329
Error: 0.000329
Error: 0.000328
Error: 0.000327
Error: 0.000326
Error: 0.000326
Error: 0.000325
Error: 0.000324
Error: 0.000323
Error: 0.000323
Error: 0.000152
Error: 0.000151
Error: 0.000151
Error: 0.000151
Error: 0.000151
Error: 0.000151
Error: 0.000151
Error: 0.000150
Error: 0.000150
Error: 0.000150
Error: 0.000097
Error: 0.000097
Error: 0.000097
Error: 0.000097
Error: 0.000096
Error: 0.000096
Error: 0.000096
Error: 0.000096
Error: 0.000096
Error: 0.000096
Error: 0.000071
Error: 0.000070
Error: 0.000070
Error: 0.000070
Error: 0.000070
Error: 0.000070
Error: 0.000070
Error: 0.000070
Error: 0.000070
Error: 0.000070


In [83]:
model.see_layers()

[[ 0.0007958   0.0002464   0.00303435]
 [ 0.00874162 -0.00544687  0.0076796 ]
 [-0.01879218  0.00795242  0.00888047]
 [-0.01104382  0.00554604 -0.01577356]
 [ 0.00111234 -0.01351017  0.00770306]
 [ 0.00092813  0.01187418 -0.00538075]
 [-0.00507397 -0.00180853  0.00479092]
 [ 0.0063072  -0.01443018 -0.02114664]
 [ 0.00874352  0.01230031 -0.02019271]
 [ 0.0094064  -0.00012175  0.01973357]]
[[-0.00046636  0.03469175 -0.00899544  0.0135808  -0.00299637  0.00943844
  -0.00678947  0.0174476  -0.00342462 -0.00701473  0.00707581]
 [-0.02381268  0.0103798  -0.00423878 -0.01344061  0.00552207  0.00073371
  -0.00176467  0.00389998  0.01278048 -0.00710427  0.00957323]
 [-0.00633139  0.01294625  0.01314058 -0.00440334  0.01440265  0.00316201
   0.00803516 -0.00196296  0.0111111   0.01300281  0.01431316]
 [ 0.00783344 -0.0044299  -0.00174888  0.00276842 -0.0004908   0.01152867
   0.00640821  0.01486623 -0.0074772  -0.00637027  0.0200963 ]
 [ 0.01468586  0.00321229  0.02879443 -0.00388339  0.0091058 